# LeafShift Week 7 data inspection

We open the images for the 10 shared tomato and potato classes in PlantVillage and PlantDoc. The class comes from the image's folder. PlantDoc's original train and test folders are counted separately and together. All counts are before cleaning.

The saved outputs were generated from [PlantVillage](https://github.com/spMohanty/PlantVillage-Dataset/tree/7f7ecc7e1eaca78107e3affe7cb5abd9427e139a) and [PlantDoc](https://github.com/pratikkayal/PlantDoc-Dataset/tree/5467f6012d78d1c446145d5f582da6096f852ae8). To rerun the notebook, follow the setup in the project README and choose **Run All**.

## 1. Locate the dataset folders

In [1]:
from collections import Counter
from pathlib import Path

from IPython.display import Markdown, display
from PIL import Image

# Works when Jupyter starts in the project root or in notebooks/.
project = Path.cwd()
if project.name == "notebooks":
    project = project.parent

data = project / "data"
plantvillage = data / "PlantVillage-Dataset" / "raw" / "color"
plantdoc = data / "PlantDoc-Dataset"

for folder in (plantvillage, plantdoc / "train", plantdoc / "test"):
    if not folder.is_dir():
        raise FileNotFoundError(f"Missing {folder}. Follow the setup in README.md.")

## 2. Match the class labels

The two sources use different names for the same project classes. Each row below gives the shared label, followed by its PlantVillage and PlantDoc folder names.

In [2]:
classes = [
    ("Tomato healthy", "Tomato___healthy", "Tomato leaf"),
    ("Tomato bacterial spot", "Tomato___Bacterial_spot", "Tomato leaf bacterial spot"),
    ("Tomato early blight", "Tomato___Early_blight", "Tomato Early blight leaf"),
    ("Tomato late blight", "Tomato___Late_blight", "Tomato leaf late blight"),
    ("Tomato leaf mold", "Tomato___Leaf_Mold", "Tomato mold leaf"),
    ("Tomato Septoria leaf spot", "Tomato___Septoria_leaf_spot", "Tomato Septoria leaf spot"),
    ("Tomato mosaic virus", "Tomato___Tomato_mosaic_virus", "Tomato leaf mosaic virus"),
    ("Tomato yellow leaf curl virus", "Tomato___Tomato_Yellow_Leaf_Curl_Virus", "Tomato leaf yellow virus"),
    ("Potato early blight", "Potato___Early_blight", "Potato leaf early blight"),
    ("Potato late blight", "Potato___Late_blight", "Potato leaf late blight"),
]

## 3. Open the images and count each class

We try to decode every file with a supported image extension. We keep its dimensions and count decoding failures separately.

In [3]:
image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tif", ".tiff", ".webp"}

def open_folder(folder):
    images = []
    unreadable = []

    if not folder.is_dir():
        raise FileNotFoundError(f"Missing class folder: {folder}")

    for path in sorted(folder.iterdir()):
        if path.is_file() and path.suffix.lower() in image_extensions:
            try:
                with Image.open(path) as image:
                    image.load()
                    width, height = image.size
                images.append((path, width, height))
            except (OSError, ValueError) as error:
                unreadable.append((path, str(error)))

    return images, unreadable

village_images = {}
doc_train_images = {}
doc_test_images = {}
village_errors = []
doc_errors = []

for label, village_folder, doc_folder in classes:
    village_images[label], errors = open_folder(plantvillage / village_folder)
    village_errors.extend(errors)

    doc_train_images[label], errors = open_folder(plantdoc / "train" / doc_folder)
    doc_errors.extend(errors)

    doc_test_images[label], errors = open_folder(plantdoc / "test" / doc_folder)
    doc_errors.extend(errors)

In [4]:
village_counts = {label: len(images) for label, images in village_images.items()}
train_counts = {label: len(images) for label, images in doc_train_images.items()}
test_counts = {label: len(images) for label, images in doc_test_images.items()}
doc_counts = {label: train_counts[label] + test_counts[label] for label in train_counts}

table = [
    "| Class | PlantVillage | PlantDoc train | PlantDoc test | PlantDoc total |",
    "|---|---:|---:|---:|---:|",
]
for label, _, _ in classes:
    table.append(
        f"| {label} | {village_counts[label]:,} | {train_counts[label]:,} | "
        f"{test_counts[label]:,} | {doc_counts[label]:,} |"
    )

table.append(
    f"| **Total** | **{sum(village_counts.values()):,}** | "
    f"**{sum(train_counts.values()):,}** | **{sum(test_counts.values()):,}** "
    f"| **{sum(doc_counts.values()):,}** |"
)
display(Markdown("\n".join(table)))

print(f"Unreadable image files — PlantVillage: {len(village_errors)}, PlantDoc: {len(doc_errors)}")
for path, reason in village_errors + doc_errors:
    print(f"{path.relative_to(data)}: {reason}")

| Class | PlantVillage | PlantDoc train | PlantDoc test | PlantDoc total |
|---|---:|---:|---:|---:|
| Tomato healthy | 1,591 | 55 | 8 | 63 |
| Tomato bacterial spot | 2,127 | 101 | 9 | 110 |
| Tomato early blight | 1,000 | 79 | 9 | 88 |
| Tomato late blight | 1,909 | 101 | 10 | 111 |
| Tomato leaf mold | 952 | 85 | 6 | 91 |
| Tomato Septoria leaf spot | 1,771 | 140 | 11 | 151 |
| Tomato mosaic virus | 373 | 44 | 10 | 54 |
| Tomato yellow leaf curl virus | 5,357 | 70 | 6 | 76 |
| Potato early blight | 1,000 | 109 | 8 | 117 |
| Potato late blight | 1,000 | 97 | 8 | 105 |
| **Total** | **17,080** | **881** | **85** | **966** |

Unreadable image files — PlantVillage: 0, PlantDoc: 0


## 4. Check data quality and labels

Look at how much the class sizes vary and check that every class is represented in both datasets.

In [5]:
village_sizes = Counter(
    (width, height)
    for images in village_images.values()
    for _, width, height in images
)

doc_small = sum(
    min(width, height) < 224
    for split in (doc_train_images, doc_test_images)
    for images in split.values()
    for _, width, height in images
)

largest = max(village_counts, key=village_counts.get)
smallest = min(village_counts, key=village_counts.get)
smallest_count = village_counts[smallest]
ratio = village_counts[largest] / smallest_count if smallest_count else float("inf")

print("PlantVillage sizes (width × height):")
for (width, height), count in sorted(village_sizes.items()):
    print(f"  {width} × {height}: {count:,} images")

print(f"\nLargest PlantVillage class: {largest} ({village_counts[largest]:,})")
print(f"Smallest PlantVillage class: {smallest} ({smallest_count:,})")
print(f"Largest-to-smallest ratio: {ratio:.1f}×")
print(f"PlantDoc images under 224 px on either side: {doc_small}")

usable_labels = all(village_counts.values()) and all(doc_counts.values())
print(f"All 10 mapped labels have readable images in both datasets: {usable_labels}")

PlantVillage sizes (width × height):
  256 × 256: 17,080 images

Largest PlantVillage class: Tomato yellow leaf curl virus (5,357)
Smallest PlantVillage class: Tomato mosaic virus (373)
Largest-to-smallest ratio: 14.4×
PlantDoc images under 224 px on either side: 65
All 10 mapped labels have readable images in both datasets: True


### View sample photos locally

Set `show_samples` to `True` and run this cell to look at examples of the same class from both sources. The saved notebook omits dataset photos.

In [6]:
show_samples = False

if show_samples:
    for label in ("Tomato early blight", "Potato late blight"):
        examples = {
            "PlantVillage": village_images[label],
            "PlantDoc": doc_train_images[label] + doc_test_images[label],
        }
        for source, images in examples.items():
            if images:
                path, _, _ = images[len(images) // 2]
                with Image.open(path) as image:
                    preview = image.convert("RGB")
                    preview.thumbnail((320, 320))
                display(Markdown(f"**{source} — {label}**"))
                display(preview)

## 5. Results for Section 3 of the handout

In [7]:
display(Markdown(
    f"- **Images and classes.** We opened {sum(village_counts.values()):,} "
    f"PlantVillage colour images and {sum(doc_counts.values()):,} PlantDoc images "
    f"from the {len(classes)} shared tomato and potato classes. The original PlantDoc "
    f"test folder contains {sum(test_counts.values()):,} of these images.\n"
    f"- **Visible quality issue.** In PlantVillage, {largest} has "
    f"{village_counts[largest]:,} images and {smallest} has {smallest_count:,}. "
    f"The largest class is about {ratio:.1f} times the size of the smallest.\n"
    f"- **Target present and usable.** {'Yes' if usable_labels else 'Needs review'}. "
    "The folder names supply the crop/disease label and the mapping aligns "
    "those labels across both sources. The next section checks the cleaned manifests."
))

- **Images and classes.** We opened 17,080 PlantVillage colour images and 966 PlantDoc images from the 10 shared tomato and potato classes. The original PlantDoc test folder contains 85 of these images.
- **Visible quality issue.** In PlantVillage, Tomato yellow leaf curl virus has 5,357 images and Tomato mosaic virus has 373. The largest class is about 14.4 times the size of the smallest.
- **Target present and usable.** Yes. The folder names supply the crop/disease label and the mapping aligns those labels across both sources. The next section checks the cleaned manifests.

## 6. Verify the post-cleaning counts

The raw counts above describe the downloaded datasets. For PlantVillage, the grouping manifest marks extra copies of identical images for exclusion. For PlantDoc, the clean manifest lists the photos retained after review. The paths are checked against the local datasets before counting.

In [8]:
from csv import DictReader

docs = project / "docs"

with (docs / "plantvillage_leaf_groups.csv").open(encoding="utf-8", newline="") as source:
    village_manifest = list(DictReader(source))

with (docs / "plantdoc_clean_manifest.csv").open(encoding="utf-8", newline="") as source:
    doc_manifest = list(DictReader(source))

# The two PlantDoc layouts use different filenames but share the same folders.
original_files = {p.relative_to(plantdoc).as_posix()
                  for split in (doc_train_images, doc_test_images)
                  for images in split.values() for p, _, _ in images}
path_column = (
    "windows_path" if all(row["windows_path"] in original_files for row in doc_manifest)
    else "original_path"
)
if not all(row[path_column] in original_files for row in doc_manifest):
    raise FileNotFoundError("The PlantDoc files do not match either manifest layout. Check README.md.")

village_files = {p.relative_to(plantvillage).as_posix()
                 for images in village_images.values() for p, _, _ in images}
if {row["image_path"] for row in village_manifest} != village_files:
    raise FileNotFoundError("The PlantVillage files do not match the audited source.")

village_retained = Counter(row["class_folder"] for row in village_manifest if row["action"] == "retain")
doc_retained = Counter(row["class_folder"] for row in doc_manifest)

table = [
    "| Shared class | PlantVillage raw | PlantVillage retained | PlantDoc raw | PlantDoc retained |",
    "|---|---:|---:|---:|---:|",
]
for label, pv_folder, pd_folder in classes:
    table.append(f"| {label} | {village_counts[label]:,} | {village_retained[pv_folder]:,} | "
                 f"{doc_counts[label]:,} | {doc_retained[pd_folder]:,} |")

table.append(f"| **Total** | **{sum(village_counts.values()):,}** | "
             f"**{sum(village_retained.values()):,}** | **{sum(doc_counts.values()):,}** | "
             f"**{sum(doc_retained.values()):,}** |")
display(Markdown("\n".join(table)))

unresolved = sum(not row["leaf_id"] for row in village_manifest)
mosaic_groups = {row["split_group_id"] for row in village_manifest
                 if row["class_folder"] == "Tomato___Tomato_mosaic_virus" and row["action"] == "retain"}
print(f"PlantVillage images without individual leaf IDs: {unresolved:,}")
print(f"PlantVillage tomato mosaic virus split groups: {len(mosaic_groups)}")

| Shared class | PlantVillage raw | PlantVillage retained | PlantDoc raw | PlantDoc retained |
|---|---:|---:|---:|---:|
| Tomato healthy | 1,591 | 1,585 | 63 | 63 |
| Tomato bacterial spot | 2,127 | 2,127 | 110 | 105 |
| Tomato early blight | 1,000 | 1,000 | 88 | 76 |
| Tomato late blight | 1,909 | 1,901 | 111 | 100 |
| Tomato leaf mold | 952 | 952 | 91 | 90 |
| Tomato Septoria leaf spot | 1,771 | 1,771 | 151 | 139 |
| Tomato mosaic virus | 373 | 373 | 54 | 53 |
| Tomato yellow leaf curl virus | 5,357 | 5,357 | 76 | 72 |
| Potato early blight | 1,000 | 1,000 | 117 | 102 |
| Potato late blight | 1,000 | 1,000 | 105 | 93 |
| **Total** | **17,080** | **17,066** | **966** | **893** |

PlantVillage images without individual leaf IDs: 4,306
PlantVillage tomato mosaic virus split groups: 1
